# 02 — PEARL-Lite từ đầu: HC3 → GPT3.5-Mixed 0% và HC3 Insert 25%

Bật **Internet + GPU** trên Kaggle, push code mới lên `version4`, rồi Run All.
Luồng chạy: **clone → chuẩn bị retention pairs → train predictor → train PEARL-GCN từ đầu → test → ZIP có seed**.

GCN khởi tạo ngẫu nhiên, predictor nạp weights vừa học từ retention pairs. Cả hai tiếp tục được tối ưu
bằng clean classification BCE + lambda_edge × valid edge BCE. Không train/nạp baseline GCN riêng.
RoBERTa pretrained vẫn đóng băng, spaCy và kiến trúc GCN giữ nguyên.

Mặc định seed **2026**, chạy toàn bộ HC3, test đúng **GPT3.5-Mixed 0%** và **HC3 Insert 25%**.
`SMOKE_TEST=True`: 24 train / 8 validation, 4 mẫu mỗi test; chỉ dùng kiểm tra thực thi.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, time, hashlib, zipfile

SEED = 2026
EDGE_EPOCHS = 20
EPOCHS = 40  # Train PEARL-GCN từ GCN khởi tạo ngẫu nhiên
EPSILON, LAMBDA_EDGE = 0.8, 0.1
BRANCH = 'version4'
REPO_URL = 'https://github.com/tphan1420/PRDetect-Research.git'
SMOKE_TEST = False
WORK = Path('/kaggle/working/pearl_scratch_hc3_gpt35_0pct_insert25')

# Optional local verification hooks; trên Kaggle không cần đặt các biến này.
SMOKE_TEST = os.environ.get('PEARL_NOTEBOOK_SMOKE', str(int(SMOKE_TEST))) == '1'
WORK = Path(os.environ.get('PEARL_NOTEBOOK_WORK', str(WORK))).resolve()
REPO = Path(os.environ.get('PEARL_EXISTING_REPO', str(WORK / 'PRDetect-Research'))).resolve()
MODEL_DIR, EVAL_DIR = WORK / f'seed{SEED}/model', WORK / f'seed{SEED}/evaluation'
RUN = WORK / f'seed{SEED}'
WORK.mkdir(parents=True, exist_ok=True)
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HOME'] = str(WORK / 'hf_cache')
print({'seed': SEED, 'branch': BRANCH, 'smoke': SMOKE_TEST, 'work': str(WORK)})


In [ ]:
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO)], check=True)
assert (REPO / 'pearl/train_gcn.py').is_file(), 'Hãy push code PEARL lên version4 trước.'
assert 'gcn_initialization' in (REPO / 'pearl/train_gcn.py').read_text(encoding='utf-8'), 'Push bản code hỗ trợ scratch lên version4 trước.'
COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
assert subprocess.check_output(['git', '-C', str(REPO), 'branch', '--show-current'], text=True).strip() == BRANCH
if os.environ.get('PEARL_SKIP_INSTALL') != '1':
    # Giữ nguyên torch CUDA của Kaggle.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-geometric>=2.5,<3',
        'transformers>=4.48,<5', 'spacy==3.8.7', 'click>=8', 'scikit-learn>=1.3'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
        'https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl'], check=True)
sys.path.insert(0, str(REPO))
os.chdir(REPO)
import torch, spacy
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print({'commit': COMMIT, 'device': DEVICE})

def run(stage, arguments):
    logs = RUN / 'logs'
    logs.mkdir(parents=True, exist_ok=True)
    command = [sys.executable, '-u', *map(str, arguments)]
    env = dict(os.environ, PYTHONHASHSEED=str(SEED), PYTHONUNBUFFERED='1')
    with (logs / f'{stage}.log').open('w', encoding='utf-8') as log:
        process = subprocess.Popen(command, cwd=REPO, env=env, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace')
        try:
            for line in process.stdout:
                log.write(line)
                print(line, end='', flush=True)
            status = process.wait()
        except BaseException:
            process.terminate()
            process.wait()
            raise
    if status:
        raise subprocess.CalledProcessError(status, command)


## Chuẩn bị HC3 và retention pairs
Chỉ HC3 train tạo dữ liệu học; validation dùng đánh giá/chọn checkpoint.
Quarantine overlap trước khi sinh edits, giữ nguyên file nguồn. Nhãn stable/changed chỉ được gán
khi hai endpoint map 1:1; Delete mất endpoint hoặc split/merge là unknown/masked.


In [ ]:
TRAIN_JSON = REPO / 'data/original_text/benchmarks/hc3/hc3_train.json'
VAL_JSON = REPO / 'data/original_text/benchmarks/hc3/hc3_val.json'
assert TRAIN_JSON.is_file() and VAL_JSON.is_file()
CONFIG = RUN / 'config.json'
assert not CONFIG.exists(), 'Run đã tồn tại. Đổi WORK để tránh ghi đè.'
RUN.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)
EVAL_DIR.mkdir(exist_ok=True)
ROBERTA = Path(os.environ.get('PEARL_ROBERTA_LOCAL', str(REPO / 'roberta-base'))).resolve()
if not (ROBERTA / 'config.json').is_file():
    run('download_roberta', ['-m', 'pearl.download_roberta', '--output', ROBERTA])
cfg = json.loads((REPO / 'configs/pearl_hc3.json').read_text(encoding='utf-8'))
cfg.update(seed=SEED, device=DEVICE, source_train=str(TRAIN_JSON), source_validation=str(VAL_JSON),
    gcn_initialization='scratch', input_dim=768, hidden_dim=256, output_dim=64,
    roberta=str(ROBERTA), local_files_only=True,
    prepared=str(RUN / 'prepared'), edge_output=str(MODEL_DIR / 'edge'), gcn_output=str(MODEL_DIR / 'pearl'),
    edge_epochs=3 if SMOKE_TEST else EDGE_EPOCHS, gcn_epochs=2 if SMOKE_TEST else EPOCHS,
    epsilon=EPSILON, lambda_edge=LAMBDA_EDGE, notebook_commit=COMMIT,
    train_limit=24 if SMOKE_TEST else None, validation_limit=8 if SMOKE_TEST else None,
    data_root=str(REPO / 'data'), shuffle_control=False,
    evaluation_entries=[
        {'table': 'table3', 'dataset': 'gpt35_mixed', 'condition': 'original',
         'path': str(REPO / 'data/original_text/benchmarks/gpt35_mixed/gpt3.5_mixed_test_split.json')},
        {'table': 'table4', 'dataset': 'hc3', 'condition': 'insert25',
         'path': str(REPO / 'data/perturbed_text/hc3/structural_noise/hc3_test_insert_25.json')}])
cfg.pop('baseline_checkpoint', None)  # Scratch không cần baseline checkpoint.
CONFIG.write_text(json.dumps(cfg, indent=2), encoding='utf-8')
run('prepare', ['-m', 'pearl.prepare', '--config', CONFIG])


## Train predictor → train PEARL-GCN từ đầu
Predictor học edge retention trước. Sau đó GCN khởi tạo ngẫu nhiên và cùng predictor được train
trên HC3 bằng clean BCE + lambda_edge × document-balanced edge BCE. Chỉ source-validation BCE
chọn checkpoint; hai tập test chưa được parse ở các bước train. Không tạo graph `.pkl` baseline.


In [ ]:
run('edge_predictor', ['-m', 'pearl.train_edge', '--config', CONFIG])
run('pearl_hc3_from_scratch', ['-m', 'pearl.train_gcn', '--config', CONFIG])
CHECKPOINT = MODEL_DIR / 'pearl/best.pt'
assert CHECKPOINT.is_file()
# Không giữ embeddings sau khi train; evaluator đọc từng text, không tạo graph .pkl test.
for name in ('train_embeddings.pt', 'validation_embeddings.pt'):
    path = RUN / 'prepared' / name
    if path.is_file():
        path.unlink()


## Test sau khi train: GPT3.5-Mixed 0% và HC3 Insert 25%
Chạy đúng 2 điều kiện, không mở các điều kiện khác. Full run: 1,000 mẫu mỗi tập;
smoke: 4 mẫu mỗi tập. Có accuracy, AUROC, F1, recall, confusion matrix và từng xác suất dự đoán.


In [ ]:
REPORT = EVAL_DIR / f'metrics_seed{SEED}.json'
arguments = ['-m', 'pearl.evaluate', '--config', CONFIG, '--output', REPORT]
if SMOKE_TEST:
    arguments += ['--limit', '4']
run('evaluate', arguments)
report = json.loads(REPORT.read_text(encoding='utf-8'))
assert len(report['results']) == 2
for result in report['results']:
    assert result['status'] == 'ok', result
    assert result['metrics']['n'] == (4 if SMOKE_TEST else 1000)
    print(json.dumps({'seed': SEED, 'dataset': result['dataset'], 'condition': result['condition'],
                      **result['metrics']}, indent=2))


In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile
metadata = {'seed': SEED, 'n_seeds': 1, 'branch': BRANCH, 'commit': COMMIT, 'smoke': SMOKE_TEST,
    'gcn_initialization': 'scratch', 'predictor_initialization': 'retention_pretrained',
    'edge_epochs': cfg['edge_epochs'], 'pearl_epochs': cfg['gcn_epochs'],
    'test_conditions': ['gpt35_mixed_0pct', 'hc3_insert25'],
    'torch': str(torch.__version__), 'spacy': spacy.__version__}
METADATA = RUN / f'run_seed{SEED}.json'
METADATA.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
ENVIRONMENT = RUN / 'pip-freeze.txt'
ENVIRONMENT.write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True), encoding='utf-8')
suffix = '_smoke' if SMOKE_TEST else ''
ZIP_PATH = WORK / f'pearl_lite_scratch_hc3_to_gpt35_0pct_hc3_insert25_seed{SEED}{suffix}.zip'
assert not ZIP_PATH.exists(), 'Không ghi đè ZIP run cũ.'
files = [CHECKPOINT, CONFIG, REPORT, METADATA, ENVIRONMENT,
    MODEL_DIR / 'edge/edge.pt', MODEL_DIR / 'edge/metrics.json', MODEL_DIR / 'pearl/training.json',
    RUN / 'prepared/manifest.json', *sorted((RUN / 'logs').glob('*.log'))]
assert all(path.is_file() for path in files)
with ZipFile(ZIP_PATH, 'w', ZIP_DEFLATED) as archive:
    for path in files:
        archive.write(path, path.relative_to(RUN))
with ZipFile(ZIP_PATH) as archive:
    assert archive.testzip() is None
print({'zip': str(ZIP_PATH), 'bytes': ZIP_PATH.stat().st_size, 'seed': SEED, 'n_seeds': 1})
from IPython.display import display, FileLink
display(FileLink(str(ZIP_PATH)))
